Język: [English](lab.en.ipynb) | **Polski**

# Laboratorium 06. Projektowanie, trenowanie i analiza CNN na KMNIST

Materiał towarzyszący [`README.pl.md`](README.pl.md) korzysta ze zbioru
KMNIST zamiast Fashion-MNIST. Eksperyment sprawdza wpływ decyzji
architektonicznych w CNN na jakość, liczbę parametrów i koszt.


In [ ]:
import os
from pathlib import Path

import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as T

torch.manual_seed(0)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Uzywane urzadzenie: {device}")

## 1. Dane, rozkład klas i kształty tensorów


In [ ]:
import contextlib
import io

_REPO_ROOT = next(
    (
        parent
        for parent in (Path.cwd(), *Path.cwd().parents)
        if (parent / "pyproject.toml").is_file()
    ),
    Path.cwd(),
)
DATA_CACHE_DIR = (
    Path(os.environ.get("DEEP_LEARNING_DATA_CACHE", _REPO_ROOT / "data_cache"))
    .expanduser()
    .resolve()
)

# Pobieranie wypisuje postep (pasek tqdm, domyslnie na stderr) co kilka
# procent - w notebooku to szum, wiec wyciszamy oba strumienie tutaj
# (plik i jego MD5 sa i tak weryfikowane przez torchvision niezaleznie
# od tego, co wypisuje na ekran).
with contextlib.redirect_stdout(io.StringIO()), contextlib.redirect_stderr(io.StringIO()):
    train_full = torchvision.datasets.KMNIST(
        root=DATA_CACHE_DIR, train=True, download=True, transform=T.ToTensor()
    )
    test_full = torchvision.datasets.KMNIST(
        root=DATA_CACHE_DIR, train=False, download=True, transform=T.ToTensor()
    )

class_names = train_full.classes
print(f"train_full: {len(train_full)} przykladow, test_full: {len(test_full)} przykladow")
print(f"ksztalt jednego obrazu: {tuple(train_full[0][0].shape)}")
print("klasy (znaki kuzushiji):", class_names)

fig, axes = plt.subplots(1, 8, figsize=(14, 2.2))
for ax, i in zip(axes, range(8)):
    img, label = train_full[i]
    ax.imshow(img[0], cmap="gray")
    ax.set_title(class_names[label], fontsize=9)
    ax.axis("off")
plt.tight_layout()
plt.show()

**Zadanie.** Za pomocą `torch.bincount` na `train_full.targets` należy
policzyć liczbę przykładów każdej z 10 klas i ocenić, czy zbiór treningowy
jest w przybliżeniu zbalansowany.


In [ ]:
# TODO: policz liczbe przykladow kazdej z 10 klas w train_full.targets
# (torch.bincount) i wypisz
raise NotImplementedError("TODO: rozklad klas")

_(Miejsce na odpowiedź.)_

## 2. Podział na train/val/test i normalizacja bez przecieku

KMNIST ma już osobny zbiór testowy (`test_full`, inny plik niż
treningowy). Zbiór walidacyjny należy wydzielić **z danych treningowych**
bez powtórzeń indeksów.


In [ ]:
def make_split(n_total, n_train, n_val, seed):
    """Zwraca (train_idx, val_idx) - rozlaczne indeksy z [0, n_total)."""
    # TODO:
    # 1. uzyj torch.randperm(n_total, generator=...) z podanym seed
    # 2. train_idx = pierwsze n_train indeksow z permutacji
    # 3. val_idx = kolejne n_val indeksow (NIE nakladajace sie z train_idx)
    raise NotImplementedError("TODO: make_split")

In [ ]:
N_TRAIN, N_VAL, N_TEST = 5000, 1000, 1500
SPLIT_SEED = 0

train_idx, val_idx = make_split(len(train_full), N_TRAIN, N_VAL, SPLIT_SEED)
assert len(set(train_idx.tolist()) & set(val_idx.tolist())) == 0, "train/val sie nakladaja!"

g = torch.Generator().manual_seed(SPLIT_SEED)
test_idx = torch.randperm(len(test_full), generator=g)[:N_TEST]

X_train_raw = torch.stack([train_full[i][0] for i in train_idx])
y_train = torch.stack([torch.tensor(train_full[i][1]) for i in train_idx])
X_val_raw = torch.stack([train_full[i][0] for i in val_idx])
y_val = torch.stack([torch.tensor(train_full[i][1]) for i in val_idx])
X_test_raw = torch.stack([test_full[i][0] for i in test_idx])
y_test = torch.stack([torch.tensor(test_full[i][1]) for i in test_idx])

print(
    f"X_train={tuple(X_train_raw.shape)}  X_val={tuple(X_val_raw.shape)}  "
    f"X_test={tuple(X_test_raw.shape)}"
)

Funkcja `normalize_with_train_stats` powinna obliczać średnią i odchylenie
standardowe pikseli **wyłącznie** na `X_train_raw`. Te same statystyki, bez
ponownego obliczania, należy zastosować do wszystkich trzech zbiorów.


In [ ]:
def normalize_with_train_stats(X_train_raw, X_val_raw, X_test_raw):
    """Zwraca (mean, std, X_train, X_val, X_test). Statystyki TYLKO z X_train_raw."""
    # TODO:
    # 1. policz mean, std WYLACZNIE z X_train_raw (skalary, cale tensory)
    # 2. znormalizuj X_train_raw, X_val_raw, X_test_raw TYMI SAMYMI
    #    statystykami (bez przeliczania osobno dla val/test!)
    # 3. zwroc (mean, std, X_train, X_val, X_test)
    raise NotImplementedError("TODO: normalizacja bez przecieku")

In [ ]:
pixel_mean, pixel_std, X_train, X_val, X_test = normalize_with_train_stats(
    X_train_raw, X_val_raw, X_test_raw
)
X_train, y_train = X_train.to(device), y_train.to(device)
X_val, y_val = X_val.to(device), y_val.to(device)
X_test, y_test = X_test.to(device), y_test.to(device)

print(f"srednia pikseli (train): {pixel_mean.item():.4f}  odchylenie std: {pixel_std.item():.4f}")
print(f"po normalizacji, X_train: mean={X_train.mean().item():.4f} std={X_train.std().item():.4f}")
assert abs(X_train.mean().item()) < 0.05
assert abs(X_train.std().item() - 1.0) < 0.05
print("OK: X_train po normalizacji ma srednia ~0 i odchylenie ~1.")

## 3. Model bazowy (gotowe)

Prosty klasyfikator liniowy nie jest CNN i nie wykorzystuje dwuwymiarowej
struktury obrazu. Jego wynik stanowi punkt odniesienia.


In [ ]:
def train_eval(model_cls, epochs, lr=0.001, batch_size=64, seed=0):
    """Konstruuje model_cls() PO ustawieniu ziarna (kontroluje init. wag,
    nie tylko kolejnosc batchy), trenuje na (X_train, y_train), zwraca
    (model, metryki)."""
    import time

    torch.manual_seed(seed)
    model = model_cls().to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    loss_fn = nn.CrossEntropyLoss()
    n = X_train.shape[0]

    start = time.time()
    for _epoch in range(epochs):
        perm = torch.randperm(n)
        for i in range(0, n, batch_size):
            idx = perm[i : i + batch_size]
            opt.zero_grad()
            out = model(X_train[idx])
            loss = loss_fn(out, y_train[idx])
            loss.backward()
            opt.step()
    train_time = time.time() - start

    model.eval()
    with torch.no_grad():
        val_acc = (model(X_val).argmax(1) == y_val).float().mean().item()
        start = time.time()
        test_logits = model(X_test)
        infer_time = time.time() - start
    test_acc = (test_logits.argmax(1) == y_test).float().mean().item()
    train_acc = (model(X_train).argmax(1) == y_train).float().mean().item()
    n_params = sum(p.numel() for p in model.parameters())

    return model, {
        "train_acc": train_acc,
        "val_acc": val_acc,
        "test_acc": test_acc,
        "train_time_s": train_time,
        "infer_time_s": infer_time,
        "n_params": n_params,
    }


class Baseline(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(nn.Flatten(), nn.Linear(28 * 28, 10))

    def forward(self, x):
        return self.net(x)


baseline_model, baseline_res = train_eval(Baseline, epochs=5, seed=0)
print("Baseline:", baseline_res)

## 4. Projekt CNN, wariant 1

Należy zaprojektować `CNNVariant1` z użyciem `nn.Conv2d`, aktywacji i
opcjonalnie `nn.MaxPool2d`. Model musi przyjmować `(N, 1, 28, 28)` i
zwracać `(N, 10)`.


In [ ]:
class CNNVariant1(nn.Module):
    """Twoj projekt: wybierz liczbe warstw konwolucyjnych, liczbe filtrow,
    rozmiar kernela, czy uzywac poolingu. Model musi przyjmowac
    (N, 1, 28, 28) i zwracac (N, 10)."""

    def __init__(self):
        super().__init__()
        # TODO: zdefiniuj warstwy (nn.Conv2d, nn.MaxPool2d, nn.Linear, ...)
        raise NotImplementedError("TODO: architektura CNNVariant1")

    def forward(self, x):
        # TODO: polacz warstwy z aktywacjami (np. torch.relu), sploszcz
        # (x.flatten(1)) przed ostatnia warstwa Linear
        raise NotImplementedError("TODO: forward CNNVariant1")

**Wymagana weryfikacja kształtów.** Przykładowy batch należy przepuścić
przez model i sprawdzić kształt tensora po każdej warstwie. Pozwala to
wykryć błędny rozmiar wejścia do `nn.Linear` po `flatten()`.


In [ ]:
def print_shapes(model, input_shape=(2, 1, 28, 28)):
    x = torch.zeros(*input_shape)
    hooks = []

    def hook(module, _inp, out):
        print(f"{module.__class__.__name__:>12s} -> {tuple(out.shape)}")

    for layer in model.children():
        hooks.append(layer.register_forward_hook(hook))
    print(f"{'wejscie':>12s} -> {tuple(x.shape)}")
    with torch.no_grad():
        model(x)
    for h in hooks:
        h.remove()


print_shapes(CNNVariant1())

**Zadanie.** Należy opisać i uzasadnić wybraną liczbę warstw i filtrów,
rozmiar kernela oraz sposób użycia poolingu.


_(Miejsce na odpowiedź.)_

## 5. Trening i walidacja wariantu 1


In [ ]:
variant1_model, variant1_res = train_eval(CNNVariant1, epochs=5, seed=0)
print("Wariant 1:", variant1_res)

## 6. Projekt wariantu 2

Drugi wariant musi różnić się od pierwszego co najmniej jednym konkretnym
i uzasadnionym elementem.


**Zadanie przed napisaniem kodu.** Należy zapisać konkretną hipotezę
sprawdzaną przez wprowadzoną zmianę.


_(Miejsce na odpowiedź.)_

In [ ]:
class CNNVariant2(nn.Module):
    """Twoj drugi projekt - musi sie roznic od Wariantu 1 w konkretny,
    nazwany sposob (np. inna liczba filtrow, kernel, obecnosc poolingu)."""

    def __init__(self):
        super().__init__()
        # TODO: zdefiniuj warstwy, inne niz w CNNVariant1
        raise NotImplementedError("TODO: architektura CNNVariant2")

    def forward(self, x):
        # TODO: forward dla Wariantu 2
        raise NotImplementedError("TODO: forward CNNVariant2")

In [ ]:
print_shapes(CNNVariant2())

## 7. Porównanie baseline i dwóch wariantów CNN

Wszystkie trzy modele należy wytrenować dla co najmniej dwóch ziaren
losowości. Zestawienie powinno zawierać accuracy walidacyjną i testową,
liczbę parametrów oraz czas treningu i inferencji. Zbiór testowy może być
użyty po raz pierwszy dopiero w tej sekcji.


In [ ]:
# TODO:
# 1. dla kazdego z trzech modeli (Baseline, CNNVariant1, CNNVariant2) i
#    kazdego z co najmniej 2 ziarnan (np. [0, 1]): wywolaj
#    train_eval(KLASA, epochs=5, seed=seed) - PRZEKAZUJ KLASE, nie
#    instancje (model()); train_eval konstruuje model PO ustawieniu
#    ziarna, inaczej seed nie kontroluje inicjalizacji wag
# 2. zapisz wyniki (val_acc, test_acc, n_params, train_time_s, ...) tak,
#    zeby moc je podsumowac per model (np. slownik model_name -> lista wynikow)
# 3. zapisz WYTRENOWANY model dla seed=0 kazdej architektury - potrzebny
#    w sekcji 8 do confusion matrix
# 4. wypisz podsumowanie: dla kazdego modelu usrednione val_acc/test_acc
#    po ziarnach, razem z liczba parametrow i czasem treningu
raise NotImplementedError("TODO: porownanie trzech modeli")

## 8. Confusion matrix i analiza błędów

Do analizy należy wybrać model z najlepszą **walidacyjną** accuracy z sekcji 7.


In [ ]:
# TODO:
# 1. wybierz model z najlepsza SREDNIA walidacyjna accuracy (po ziarnach)
#    z all_results/trained_models z sekcji 7
# 2. policz predykcje tego modelu na X_test (model.eval(), torch.no_grad())
# 3. zbuduj confusion matrix 10x10 (wiersze=prawdziwa klasa, kolumny=
#    przewidziana) - mozna recznie w petli po (y_true, y_pred) albo
#    torch.zeros(10,10) + inkrementacja
# 4. narysuj confusion matrix (plt.imshow) z etykietami klas na obu osiach
# 5. znajdz indeksy zle sklasyfikowanych przykladow testowych, wyswietl
#    6-8 z nich (obraz + prawdziwa/przewidziana etykieta w tytule)
raise NotImplementedError("TODO: confusion matrix i analiza bledow")

_(Miejsce na odpowiedź.)_

## 9. Wnioski

Odpowiedzi należy uzasadnić rzeczywistymi wynikami z sekcji 7 i 8.


**1. Który wariant CNN wygrał na walidacji? Czy to ten sam, który wygrał na teście?**

_(Miejsce na odpowiedź.)_

**2. Czy wariant z większą liczbą parametrów był szybszy czy wolniej się trenował? Zgodne z intuicją "mniej parametrów = mniej obliczeń", czy ją podważa?**

_(Miejsce na odpowiedź.)_

**3. Czy wyższa accuracy treningowa najlepszego modelu oznacza dobrą generalizację? Na jakich wynikach można oprzeć ocenę?**

_(Miejsce na odpowiedź.)_

**4. Jakie jest co najmniej jedno ograniczenie tego eksperymentu, które nie pozwala uogólnić wniosków na "Wariant X jest zawsze lepszy"?**

_(Miejsce na odpowiedź.)_